<a href="https://colab.research.google.com/github/DataFriend101/Machine_Learning/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



```
# One row represents one content page for one client on one report date. For this assignment, I analyze data from April 2026.
```



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


```
Features:
- impressions_90d -> Known before deciding whether to refresh the page.
- ctr -> Click-through rate available before the decision
- avg_position -> Search ranking available before the decision
- content_age_days -> Age of the page is already known
- days_since_last_update -> Page's update history is already known

Labels:
- Label (proxy): Pages that show signs of declining performance and may benefit from being refreshed

Context:
- client_hash_id
- content_type
- position_tier
- report_date

Excluded:
- Future clicks (excluded since they are only known after the prediction time and would cause data leakage)
- Future impressions (excluded since they contain future information that we don't have at the desicion moment)
- Future ranking position (excluded since it is observed after the decision and would leak the outcome)

```

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# Load the database
%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [11]:
# Verify the time window for the month of April
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-04-01'
  AND report_date < DATE '2026-05-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,start_date,end_date
0,10424730,2026-04-01,2026-04-30




```
# There is 10424730 rows from April 1 to the end of the month
```



In [12]:
# Verify the grain
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS n
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-04-01'
  AND report_date < DATE '2026-05-01'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n




```
# No duplicate rows were found! Which means that one row represents one content page for one client on one report date
```

In [13]:
# Verify it's availability
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-04-01'
  AND report_date < DATE '2026-05-01'
  AND ga4_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,524291



```
# After filtering to rows where ga4_data_available IS TRUE 524291 rows remain
```



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


```
# This data cannot explain everything about content performance. Different clients have different amounts of history,
so some pages may have less information available than others. Some metrics may also be unavailable for certain periods,
meaning a missing value does not always mean zero activity. And this data shows patterns and relationships,
but it cannot prove that refreshing a page directly caused better performance.
```


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.